# CyVach Phase 5, Stage C: final config x 5 fresh seeds (val only)

Runtime -> T4 GPU. Run the cells in order. Cell 2: upload the same `cyvach_phase5_bundle.zip` as before.
Each seed downloads one zip, `seed_N.zip` (about 250 MB: model + val logits for temperature scaling). Put all 5 in `D:\CYVACH_Model\colab\`.
If Chrome asks to allow multiple downloads, click **Allow**.

In [ ]:
!nvidia-smi -L

In [ ]:
from google.colab import files
import zipfile, os
up = files.upload()
zipfile.ZipFile(next(iter(up))).extractall('/content/cyvach')
os.chdir('/content/cyvach')
print(open('CODE_VERSION').read())

In [ ]:
!pip -q install -r colab_requirements.txt

In [ ]:
import subprocess, os
from google.colab import files
os.chdir('/content/cyvach')
CFG = '--balance S0 --digits D1 --placeholders U1 --synthetic-ratio 0 --lr 5e-5 --epochs 5 --max-len 128 --label-smoothing 0 --batch 32'.split()
for seed in [1, 2, 3, 4, 5]:
    out = subprocess.run(['python', 'src/train.py', '--seed', str(seed)] + CFG, capture_output=True, text=True)
    if out.returncode != 0:
        print(out.stderr[-3000:]); raise SystemExit(f'seed {seed} failed')
    run_id = out.stdout.strip().splitlines()[-1]
    print(seed, out.stdout.strip().splitlines()[-2], run_id, flush=True)
    subprocess.run(['zip', '-qr', f'seed_{seed}.zip', f'models/{run_id}', f'results/{run_id}.json', f'logs/{run_id}_epochs.csv'], check=True)
    files.download(f'seed_{seed}.zip')